In [ ]:
#  1 — Install & Import
!pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings, time
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics         import (accuracy_score, f1_score,
                                      classification_report, confusion_matrix)
print('Libraries ready')

In [ ]:
# 2 — Load & Balance Check
df    = pd.read_csv('/content/dataset_wide_balanced.csv')
total = len(df)
print(f'Rows    : {total:,}')
print(f'Columns : {len(df.columns)}')
print()

N_CLASSES = df['weakest_sub'].nunique()
dist  = df['weakest_sub'].value_counts().sort_index()
ideal = total / N_CLASSES
mx    = dist.max(); mn = dist.min()

print('── Weakest sub distribution ──────────────')
for sub, cnt in dist.items():
    flag = '\u2705' if abs(cnt-ideal)/ideal < 0.05 else '\u26a0\ufe0f'
    print(f'  {flag} {sub:<28} {cnt:>4} ({cnt/total*100:.1f}%)')
print(f'  Imbalance ratio : {mx/mn:.2f}x')
print(f'  VERDICT : {"\u2705 BALANCED" if mx/mn < 1.1 else "\u26a0\ufe0f IMBALANCED"}')
print()

# Balance chart
fig, ax = plt.subplots(figsize=(10, 4))
dist.plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
ax.axhline(ideal, color='red', linestyle='--', linewidth=1.5,
           label=f'Ideal = {ideal:.0f}')
ax.set_title('Weakest Sub Distribution \u2014 Balanced \u2705', fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('balance_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Data loaded and balanced')

In [ ]:
#  3 — Features Setup

SUB_LIST = ['Addition','Comparing Fractions','Division','Fraction to Decimal',
            'Multiplication','Ordering Fractions',
            'Simplifying Fractions','Subtraction']

FEATURES = []
for sub in SUB_LIST:
    code = sub.lower().replace(' ','_')
    FEATURES += [f'{code}_score', f'{code}_ratio', f'{code}_correct']

print(f'Features ({len(FEATURES)}):')
for f in FEATURES:
    print(f'  {f}')
print()

sub_enc = LabelEncoder()
y = sub_enc.fit_transform(df['weakest_sub'])
print('Label encodings:')
for i, cls in enumerate(sub_enc.classes_):
    print(f'  {i} = {cls}')

X = df[FEATURES].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

print(f'
Train : {len(X_tr):,}  |  Test : {len(X_te):,}')
print('Features and labels ready')

In [ ]:
#  4 — Train 4 Models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_configs = [
    {
        'name'  : 'K-Nearest Neighbors',
        'short' : 'K-Nearest\nNeighbors',
        'model' : KNeighborsClassifier(n_jobs=-1),
        'params': {
            'n_neighbors': [3, 5, 7, 11, 15],
            'weights'    : ['uniform', 'distance'],
            'metric'     : ['euclidean', 'manhattan'],
        },
        'scale' : True,
        'color' : '#3498db',
    },
    {
        'name'  : 'Logistic Regression',
        'short' : 'Logistic\nRegression',
        'model' : LogisticRegression(
                      class_weight='balanced', max_iter=2000, random_state=42),
        'params': {
            'C'     : [0.01, 0.1, 1.0, 10.0],
            'solver': ['lbfgs', 'saga'],
        },
        'scale' : True,
        'color' : '#e74c3c',
    },
    {
        'name'  : 'Gradient Boosting',
        'short' : 'Gradient\nBoosting',
        'model' : GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators' : [100, 200],
            'max_depth'    : [3, 5, 7],
            'learning_rate': [0.05, 0.1, 0.2],
        },
        'scale' : False,
        'color' : '#e67e22',
    },
    {
        'name'  : 'Random Forest',
        'short' : 'Random\nForest',
        'model' : RandomForestClassifier(
                      class_weight='balanced', random_state=42, n_jobs=-1),
        'params': {
            'n_estimators'     : [100, 200, 300],
            'max_depth'        : [5, 10, 15, None],
            'min_samples_split': [2, 5],
        },
        'scale' : False,
        'color' : '#2ecc71',
    },
]

results = {}
print('Training models...\n')
print(f'{"Model":<22} {"Accuracy":>10} {"F1 Macro":>10} {"CV Score":>10}')
print('\u2500' * 58)

for cfg in model_configs:
    name = cfg['name']
    Xtr  = X_tr_sc if cfg['scale'] else X_tr
    Xte  = X_te_sc if cfg['scale'] else X_te
    t0   = time.time()

    gs = GridSearchCV(cfg['model'], cfg['params'],
                      cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
    gs.fit(Xtr, y_tr)

    clf  = gs.best_estimator_
    pred = clf.predict(Xte)
    acc  = accuracy_score(y_te, pred)
    f1   = f1_score(y_te, pred, average='macro')

    results[name] = {
        'clf'   : clf,  'pred': pred,
        'acc'   : acc,  'f1'  : f1,
        'cv'    : gs.best_score_,
        'params': gs.best_params_,
        'cm'    : confusion_matrix(y_te, pred),
        'color' : cfg['color'],
        'short' : cfg['short'],
        'time'  : time.time() - t0,
    }
    print(f'{name:<22} {acc*100:>9.2f}% {f1*100:>9.2f}% '
          f'{gs.best_score_*100:>9.2f}%  ({results[name]["time"]:.0f}s)')

sorted_names = sorted(results, key=lambda x: results[x]['acc'], reverse=True)
best_name    = sorted_names[0]
print()
print(f'Best Model : {best_name}  ({results[best_name]["acc"]*100:.2f}%)')
print(f'(Random chance = {100/N_CLASSES:.1f}% for {N_CLASSES} classes)')
print()
print('All 4 models trained!')

In [ ]:
#  5 — Chart
order = ['K-Nearest Neighbors','Logistic Regression',
         'Gradient Boosting','Random Forest']
rank_labels = ['Best \u2605','2nd','3rd','4th']

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle('Grade 7 Math \u2014 Weakest Topic Predictor: 4 Algorithms Compared',
             fontsize=14, fontweight='bold', y=1.01)

for idx, name in enumerate(order):
    res   = results[name]
    ax    = axes[idx]
    color = res['color']
    rank  = rank_labels[sorted_names.index(name)]

    vals = [res['acc']*100, res['f1']*100, res['cv']*100]
    bars = ax.bar(['Accuracy','F1 Score','CV Score'], vals,
                  color=color, alpha=0.85, edgecolor='white', width=0.5)

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.5,
                f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)

    ax.set_ylim(0, 115)
    ax.set_title(res['short'], fontweight='bold', fontsize=12)
    ax.set_ylabel('Score (%)' if idx==0 else '')
    ax.axhline(100/N_CLASSES, color='gray', linestyle='--', alpha=0.4,
               linewidth=1, label=f'Random ({100/N_CLASSES:.1f}%)')
    ax.set_xlabel(rank, fontsize=11, color=color, fontweight='bold')
    if idx == 0: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

In [ ]:
# 6 — Summary + Confusion Matrix
print('=' * 58)
print('              FINAL RESULTS')
print('=' * 58)
print(f'{"Model":<22} {"Accuracy":>10} {"F1 Macro":>10} {"CV Score":>10}')
print('\u2500' * 58)
for name in sorted_names:
    r    = results[name]
    mark = ' \u2190 BEST' if name==best_name else ''
    print(f'{name:<22} {r["acc"]*100:>9.2f}% '
          f'{r["f1"]*100:>9.2f}% '
          f'{r["cv"]*100:>9.2f}%{mark}')
print('=' * 58)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Best Model: {best_name}', fontsize=13, fontweight='bold')

sns.heatmap(results[best_name]['cm'], annot=True, fmt='d', cmap='Blues',
            xticklabels=sub_enc.classes_,
            yticklabels=sub_enc.classes_, ax=axes[0],
            annot_kws={'size':9}, linewidths=0.3)
axes[0].set_title(
    f'Confusion Matrix \u2014 {results[best_name]["acc"]*100:.1f}%',
    fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(axes[0].get_yticklabels(), rotation=0,  fontsize=8)

accs  = [results[n]['acc']*100 for n in order]
cols  = [results[n]['color']   for n in order]
b2    = axes[1].bar(['KNN','LR','GB','RF'], accs,
                    color=cols, width=0.5, edgecolor='white')
for bar, v in zip(b2, accs):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[1].set_ylim(0, 110)
axes[1].axhline(100/N_CLASSES, color='gray', linestyle='--',
                alpha=0.5, label=f'Random ({100/N_CLASSES:.1f}%)')
axes[1].set_title('Accuracy Comparison', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBest params: {results[best_name]["params"]}')
print()
print(classification_report(y_te, results[best_name]['pred'],
                             target_names=sub_enc.classes_))

In [ ]:
# 7 — Save & Download
best_clf = results[best_name]['clf']

with open('model_best.pkl',    'wb') as f: pickle.dump(best_clf, f)
with open('model_scaler.pkl',  'wb') as f: pickle.dump(scaler,   f)
with open('model_sub_enc.pkl', 'wb') as f: pickle.dump(sub_enc,  f)

meta = {
    'best_model' : best_name,
    'accuracy'   : results[best_name]['acc'],
    'f1_macro'   : results[best_name]['f1'],
    'best_params': results[best_name]['params'],
    'features'   : FEATURES,
    'classes'    : list(sub_enc.classes_),
    'all_results': {n: {'acc': results[n]['acc'],
                        'f1' : results[n]['f1']}
                    for n in results},
}
with open('model_meta.pkl', 'wb') as f: pickle.dump(meta, f)

print(f'Best Model : {best_name}')
print(f'Accuracy   : {results[best_name]["acc"]*100:.2f}%')
print(f'F1 Macro   : {results[best_name]["f1"]*100:.2f}%')
print()
print('Ranking:')
for i, n in enumerate(sorted_names, 1):
    print(f'  {i}. {n:<22} {results[n]["acc"]*100:.2f}%')
print()

from google.colab import files
for fname in ['model_best.pkl','model_scaler.pkl','model_sub_enc.pkl',
              'model_meta.pkl','balance_check.png',
              'model_comparison.png','model_summary.png']:
    files.download(fname)
    print(f'  \u2705 {fname}')
print('\nDone!')

In [ ]:
# 8 — DEMO: Predict weakness for a new student
print("=" * 60)
print("   AI WEAKNESS DETECTION DEMO")
print("=" * 60)

demo_student = {
    "Addition"             : {"score": 18, "correct": 4},
    "Subtraction"          : {"score": 16, "correct": 3},
    "Multiplication"       : {"score":  4, "correct": 1},
    "Division"             : {"score": 14, "correct": 3},
    "Ordering Fractions"   : {"score":  6, "correct": 1},
    "Comparing Fractions"  : {"score": 14, "correct": 3},
    "Simplifying Fractions": {"score": 16, "correct": 4},
    "Fraction to Decimal"  : {"score": 20, "correct": 5},
}

row = []
for sub in SUB_LIST:
    d     = demo_student.get(sub, {"score": 10, "correct": 2})
    total = d["score"]
    corr  = d["correct"]
    ratio = round(total/20, 2)
    row  += [total, ratio, corr]

row_arr = np.array([row])
row_sc  = scaler.transform(row_arr)

pred_idx  = best_clf.predict(row_sc)[0]
pred_sub  = sub_enc.inverse_transform([pred_idx])[0]
proba     = best_clf.predict_proba(row_sc)[0]
pred_prob = proba[pred_idx] * 100

print()
print(f"{'Sub-Category':<25} {'Score':>6}  {'Correct':>7}  Result")
print("-" * 55)
for sub, d in demo_student.items():
    flag = "WEAK" if sub == pred_sub else "OK"
    print(f"{sub:<25} {d['score']:>5}/20  {d['correct']:>5}/5   {flag}")
print("-" * 55)
print(f"
AI Predicted Weakest  : {pred_sub}")
print(f"Confidence            : {pred_prob:.1f}%")
score_weak = min(demo_student, key=lambda x: demo_student[x]['score'])
print(f"Score-based weakest   : {score_weak}")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 5))
subs   = sub_enc.classes_
colors = ["#e74c3c" if s == pred_sub else "#3498db" for s in subs]
bars   = ax.barh(subs, proba * 100, color=colors, edgecolor="white")
for bar, v in zip(bars, proba * 100):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=10)
ax.set_xlabel("Weak Probability (%)")
ax.set_title(f"AI Weakness Detection - Predicted: {pred_sub}", fontweight="bold", fontsize=13)
ax.axvline(x=100/N_CLASSES, color="gray", linestyle="--", alpha=0.5, label=f"Random chance ({100/N_CLASSES:.1f}%)")
ax.legend()
plt.tight_layout()
plt.savefig("demo_result.png", dpi=150, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("demo_result.png")
print("
Demo complete!")